In [ ]:
from dolfinx import log, default_scalar_type
from dolfinx.fem.petsc import NonlinearProblem
import pyvista
import numpy as np
import ufl

from mpi4py import MPI
from dolfinx import fem, mesh, plot

L = 20.0
domain = mesh.create_box(
    MPI.COMM_WORLD, [[0.0, 0.0, 0.0], [L, 1, 1]], [20, 5, 5], mesh.CellType.hexahedron
)

# L = 1.0
# domain = mesh.create_box(
#     MPI.COMM_WORLD, [[0.0, 0.0, 0.0], [L, 1, 1]], [1, 1, 1], mesh.CellType.hexahedron
# )

In [ ]:
def left(x):
    return np.isclose(x[0], 0)


def right(x):
    return np.isclose(x[0], L)


fdim = domain.topology.dim - 1
left_facets = mesh.locate_entities_boundary(domain, fdim, left)
right_facets = mesh.locate_entities_boundary(domain, fdim, right)

In [ ]:
marked_facets = np.hstack([left_facets, right_facets])
marked_values = np.hstack([np.full_like(left_facets, 1), np.full_like(right_facets, 2)])
sorted_facets = np.argsort(marked_facets)
facet_tag = mesh.meshtags(
    domain, fdim, marked_facets[sorted_facets], marked_values[sorted_facets]
)

In [ ]:
u_bc = np.array((0,) * domain.geometry.dim, dtype=default_scalar_type)

# Weak form

In [ ]:
# V = fem.functionspace(domain, ("Lagrange", 2, (domain.geometry.dim,)))

import basix.ufl
el_u = basix.ufl.element("Lagrange", domain.basix_cell(), 2, shape=(domain.geometry.dim,))
el_p = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
el_mixed = basix.ufl.mixed_element([el_u, el_p])
W = fem.functionspace(domain, el_mixed)

w = fem.Function(W)

In [ ]:
# left_dofs = fem.locate_dofs_topological(V, facet_tag.dim, facet_tag.find(1))
# bcs = [fem.dirichletbc(u_bc, left_dofs, V)]




V_u = W.sub(0) # displacement subspace 
V_uCollapsed, V_uCollapsed_to_Vu = V_u.collapse()

u_D = fem.Function(V_uCollapsed)

left_dofs = fem.locate_dofs_topological(V=(V_u, V_uCollapsed), entity_dim=domain.topology.dim - 1, entities=facet_tag.find(1))
bcs = [fem.dirichletbc(u_D, left_dofs, V_u)]

In [ ]:
B = fem.Constant(domain, default_scalar_type((0, 0, 0)))
T = fem.Constant(domain, default_scalar_type((0, 0, 0)))

In [ ]:

# v = ufl.TestFunction(V)
# u = fem.Function(V)

# u, p = ufl.TrialFunctions(W)
(u, p) = ufl.split(w)
v_u, v_p = ufl.TestFunctions(W)


In [ ]:
# Spatial dimension
d = len(u)

# Identity tensor
I = ufl.variable(ufl.Identity(d))

# Deformation gradient
F = ufl.variable(I + ufl.grad(u))

# Right Cauchy-Green tensor
C = ufl.variable(F.T * F)

# Invariants of deformation tensors
I_1 = ufl.variable(ufl.tr(C))
J = ufl.variable(ufl.det(F))
I_1_bar = ufl.variable(J**(-2/3) * I_1) 

In [ ]:
E = default_scalar_type(1.0e4)
nu = default_scalar_type(0.3)
mu = fem.Constant(domain, E / (2 * (1 + nu)))
lmbda = fem.Constant(domain, E * nu / ((1 + nu) * (1 - 2 * nu)))

# psi = (mu / 2) * (I_1 - 3) - mu * ufl.ln(J) + (lmbda / 2) * (ufl.ln(J)) ** 2 # compressible 
psi = (mu / 2) * (I_1_bar - 3) + p*(J-1) # compressible , mixed with p = -kappa (J-1)
# psi = (mu / 2) * (I_1 - 3) # incompressible
P = ufl.diff(psi, F)

# To illustrate the difference between linear and hyperelasticity, the following lines can be uncommented to solve the linear elasticity problem.
# P = 2.0 * mu * ufl.sym(ufl.grad(u)) + lmbda * ufl.tr(ufl.sym(ufl.grad(u))) * I

Define the variational form with traction integral over all facets with value 2.
We set the quadrature degree for the integrals to 4.

In [ ]:
metadata = {"quadrature_degree": 4}
ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tag, metadata=metadata)
dx = ufl.Measure("dx", domain=domain, metadata=metadata)

In [ ]:
# residual = (
#     ufl.inner(ufl.grad(v_u), P) * dx - ufl.inner(v_u, B) * dx - ufl.inner(v_u, T) * ds(2) + ufl.inner((p+1), v_p)*dx + ufl.inner((J-1), v_p)*dx
# )

residual = (
    ufl.inner(ufl.grad(v_u), P) * dx - ufl.inner(v_u, B) * dx - ufl.inner(v_u, T) * ds(2) + ufl.inner((J-1), v_p)*dx
)

# residual = (
#     ufl.inner(ufl.grad(v_u), P) * dx - ufl.inner(v_u, B) * dx - ufl.inner(v_u, T) * ds(2) + ufl.inner(p*(J-1), v_p) * dx
# )

# residualVascularBiomechanics = (
#     ufl.inner(ufl.grad(v_u), P) * dx - ufl.inner(v_u, B) * dx - ufl.inner(v_u, T) * ds(2) + ufl.inner((J-1), v_p) * dx
# ) # ????????

In [ ]:
pass

# Solving

In [ ]:
petsc_options = {
    "snes_type": "newtonls",
    "snes_linesearch_type": "none",
    "snes_monitor": None,
    "snes_atol": 1e-8,
    "snes_rtol": 1e-8,
    "snes_stol": 1e-8,
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
}
problem = NonlinearProblem(
    residual,
    w,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="hyperelasticity",
)

In [ ]:
# plotter = pyvista.Plotter()
# plotter.open_gif("deformation.gif", fps=3)

# topology, cells, geometry = plot.vtk_mesh(u.function_space)
# function_grid = pyvista.UnstructuredGrid(topology, cells, geometry)

# values = np.zeros((geometry.shape[0], 3))
# values[:, : len(u)] = u.x.array.reshape(geometry.shape[0], len(u))
# function_grid["u"] = values
# function_grid.set_active_vectors("u")

# # Warp mesh by deformation
# warped = function_grid.warp_by_vector("u", factor=1)
# warped.set_active_vectors("u")

# # Add mesh to plotter and visualize
# actor = plotter.add_mesh(warped, show_edges=True, lighting=False, clim=[0, 10])

# # Compute magnitude of displacement to visualize in GIF
# Vs = fem.functionspace(domain, ("Lagrange", 2))
# magnitude = fem.Function(Vs)
# us = fem.Expression(
#     ufl.sqrt(sum([u[i] ** 2 for i in range(len(u))])), Vs.element.interpolation_points
# )
# magnitude.interpolate(us)
# warped["mag"] = magnitude.x.array

In [ ]:
log.set_log_level(log.LogLevel.INFO)
tval0 = -1.5 * 0.01
for n in range(1, 2):
    print(f"Time step {n}")
    T.value[2] = n * tval0
    problem.solve()
    converged = problem.solver.getConvergedReason()
    num_its = problem.solver.getIterationNumber()
    # assert converged > 0, f"Solver did not converge with reason {converged}. num_its: {num_its}"
    print(f"Solver convergence: {converged}. Number of iterations {num_its}, Load {T.value}")

#     function_grid["u"][:, : len(u)] = u.x.array.reshape(geometry.shape[0], len(u))
#     magnitude.interpolate(us)
#     warped.set_active_scalars("mag")
#     warped_n = function_grid.warp_by_vector(factor=1)
#     warped.points[:, :] = warped_n.points
#     warped.point_data["mag"][:] = magnitude.x.array
#     plotter.update_scalar_bar_range([0, 10])
#     plotter.write_frame()
# plotter.close()



In [ ]:
def petsc2array(v):
    s=v.getValues(range(0, v.getSize()[0]), range(0,  v.getSize()[1]))
    return s

In [ ]:
pass
# problem.solver.ksp.mat_op.view()
X_back = petsc2array(problem.solver.ksp.mat_op)

In [ ]:
# import numpy 
# with numpy.printoptions(threshold=numpy.inf, linewidth=200000):
#     print(np.array(X_back))

In [ ]:
# import matplotlib.pylab as plt
# plt.spy(X_back)
# plt.show()

# Postprocessing

In [ ]:
# xdmf.close()

In [ ]:

V_u_out = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1, shape=(3,)))
u_out = fem.Function(V_u_out)
u_out.name = "u"

V_p_out = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1))
p_out = fem.Function(V_p_out)
p_out.name = "p"

t=0
u_out.interpolate(w.sub(0).collapse())
p_out.interpolate(w.sub(1).collapse())

# J
V_post = fem.functionspace(domain, ("Lagrange", 1))
J_post = fem.Expression(ufl.det(I + ufl.grad(w.sub(0).collapse())), V_post.element.interpolation_points)
J_out = fem.Function(V_post)
J_out.name = "J"
J_out.interpolate(J_post)

# sigma 
# V_post_tensor = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1, shape=(3,3)))
# sigma_post = fem.Expression(I, V_post_tensor.element.interpolation_points)
# sigma_out = fem.Function(V_post_tensor)
# sigma_out.name = "sigma"
# sigma_out.interpolate(sigma_post)


In [ ]:
from pathlib import Path
from dolfinx import io
folder = Path("results")
folder.mkdir(exist_ok=True, parents=True)
xdmf = io.XDMFFile(MPI.COMM_WORLD, folder/"dokkenHyperTutIncompressible.xdmf", "w")
xdmf.write_mesh(domain)
# xdmf.write_meshtags(facet_tags, domain.geometry)
xdmf.write_function(p_out, t)
xdmf.write_function(u_out, t)
xdmf.write_function(J_out, t)
# xdmf.write_function(sigma_out, t)


xdmf.close()

In [ ]:
# #Save solution to file in XDMF format
# from pathlib import Path
# from dolfinx import io

# results_folder = Path("results")
# results_folder.mkdir(parents=True, exist_ok=True)
# with io.XDMFFile(domain.comm, results_folder / "poisson.xdmf", "w") as file:
#     file.write_mesh(domain)
#     file.write_function(u)

In [ ]:
# import pyvista

# cells, types, x = plot.vtk_mesh(domain)
# grid = pyvista.UnstructuredGrid(cells, types, x)
# grid.point_data["u"] = u.x.array.real
# grid.set_active_scalars("u")
# plotter = pyvista.Plotter()
# plotter.add_mesh(grid, show_edges=True)
# # plotter.view_xy()
# plotter.show()